In [21]:
!playwright install



Removing unused browser at /Users/lundenmandigo/Library/Caches/ms-playwright/chromium-1187
Removing unused browser at /Users/lundenmandigo/Library/Caches/ms-playwright/chromium_headless_shell-1187
Removing unused browser at /Users/lundenmandigo/Library/Caches/ms-playwright/firefox-1490
Removing unused browser at /Users/lundenmandigo/Library/Caches/ms-playwright/webkit-2203
(node:93221) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
162.3 MiB [                    ] 0% 0.0s162.3 MiB [                    ] 0% 44.4s162.3 MiB [                    ] 0% 40.4s162.3 MiB [                    ] 0% 22.1s162.3 MiB [                    ] 0% 13.0s162.3 MiB [                    ] 1% 8.0s162.3 MiB [                    ] 1% 5.7s162.3 MiB [                    ] 2% 4.9

In [22]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import asyncio

async def scrape_livesports():
    url = "https://www.livesportsontv.com/"
    
    # Switch to async_playwright
    async with async_playwright() as p:
        print("Launching browser...")
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)

        print("Scrolling to load all events. This may take a few seconds...")
        # Await the evaluate and wait functions
        last_height = await page.evaluate("document.body.scrollHeight")
        
        while True:
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await page.wait_for_timeout(2000) 
            
            new_height = await page.evaluate("document.body.scrollHeight")
            if new_height == last_height:
                break 
            last_height = new_height

        html_content = await page.content()
        await browser.close()
        print("Finished scrolling. Parsing data...")

    # The rest of the BeautifulSoup parsing remains exactly the same!
    soup = BeautifulSoup(html_content, 'html.parser')
    all_games = []

    active_date_element = soup.find('a', class_='calendar__item--active')
    game_date = active_date_element.get('id') if active_date_element else "Date Not Found"

    event_containers = soup.find_all('div', class_='events')

    for container in event_containers:
        league_tag = container.find_previous_sibling('a', href=lambda href: href and '/league/' in href)
        if league_tag and league_tag.get_text(strip=True):
            league = league_tag.get_text(strip=True)
        elif league_tag:
            league = league_tag.get('href').split('/')[-1].upper().replace('-', ' ')
        else:
            fallback = container.find_previous('a', href=lambda href: href and '/league/' in href)
            league = fallback.get_text(strip=True) if fallback else "Unknown League"

        game_rows = container.find_all('div', class_='event--wrapp')

        for row in game_rows:
            try:
                time_div = row.find('div', class_='event__info--time')
                event_time = time_div.find('time').get_text(strip=True) if time_div and time_div.find('time') else "Time Not Found"

                match_info = row.find('div', class_='event__matchInfo')
                match_link = match_info.find('a') if match_info else None
                teams = match_link.get_text(separator=" ", strip=True) if match_link else ""
                
                if not teams and match_link:
                    slug = match_link.get('href', '')
                    teams = slug.split('/')[-1].rsplit('-', 1)[0].replace('-', ' ').title()

                channels = []
                channel_list = row.find('ul', class_='event__tags')
                
                if channel_list:
                    for channel_link in channel_list.find_all('a'):
                        channel_name = channel_link.get('aria-label')
                        if channel_name:
                            channels.append(channel_name)

                all_games.append({
                    "Date": game_date,
                    "Time": event_time,
                    "League": league,
                    "Matchup": teams or "Unknown Matchup",
                    "Services": ", ".join(channels)
                })
                
            except Exception as e:
                continue

    df = pd.DataFrame(all_games)
    print(df.head())
    df.to_csv("livesports_schedule_full.csv", index=False)
    print(f"\nSuccessfully pulled {len(all_games)} games!")

# In a Jupyter Notebook, you call an async function like this:
await scrape_livesports()

Launching browser...
Scrolling to load all events. This may take a few seconds...
Finished scrolling. Parsing data...
        Date     Time                     League  \
0  2/23/2026  7:00 pm                       NBA>   
1  2/23/2026  8:00 pm                       NBA>   
2  2/23/2026  9:30 pm                       NBA>   
3  2/23/2026  7:00 pm  Men's College Basketball>   
4  2/23/2026  7:00 pm  Men's College Basketball>   

                                Matchup  \
0   San Antonio Spurs @ Detroit Pistons   
1  Sacramento Kings @ Memphis Grizzlies   
2           Utah Jazz @ Houston Rockets   
3           Louisville @ North Carolina   
4           Texas A&M-CC @ SE Louisiana   

                                            Services  
0       Peacock, FanDuel Sports Network Detroit, TSN  
1  NBA League Pass, FanDuel Sports Network Southe...  
2                   Peacock, Space City Home Network  
3  Fubo Sports, ESPN Unlimited, ESPN, ESPN App, TSN2  
4              ESPN Select, ESPN Un

In [25]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import asyncio

async def scrape_livesports():
    base_url = "https://www.livesportsontv.com"
    all_games = []
    
    async with async_playwright() as p:
        print("Launching browser...")
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        # 1. Go to the main page to find the calendar links
        await page.goto(base_url)
        print("Fetching calendar links...")
        
        # Execute quick JavaScript to grab the 'href' from the first 7 calendar items
        hrefs = await page.evaluate('''() => {
            const links = Array.from(document.querySelectorAll('a.calendar__item'));
            return links.slice(0, 7).map(a => a.getAttribute('href'));
        }''')
        
        print(f"Found {len(hrefs)} days to scrape. Starting loop...\n")

        # 2. Loop through each of the 7 days
        for i, href in enumerate(hrefs):
            # Construct the full URL for the day (e.g., base_url + "/date/tomorrow")
            day_url = f"{base_url}{href}" if href.startswith('/') else f"{base_url}/{href}"
            print(f"--- Scraping Day {i+1} of 7: {day_url} ---")
            
            # Navigate to the specific day's page
            await page.goto(day_url)

            # Scroll to load all dynamic content for THIS day
            last_height = await page.evaluate("document.body.scrollHeight")
            while True:
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await page.wait_for_timeout(2000) 
                
                new_height = await page.evaluate("document.body.scrollHeight")
                if new_height == last_height:
                    break 
                last_height = new_height

            # Grab the fully loaded HTML for this specific day
            html_content = await page.content()
            soup = BeautifulSoup(html_content, 'html.parser')

            # Extract the Date
            active_date_element = soup.find('a', class_='calendar__item--active')
            game_date = active_date_element.get('id') if active_date_element else "Date Not Found"
            print(f"Extracting games for date: {game_date}")

            # Extract the events using our trusted BeautifulSoup logic
            event_containers = soup.find_all('div', class_='events')

            for container in event_containers:
                league_tag = container.find_previous_sibling('a', href=lambda href: href and '/league/' in href)
                if league_tag and league_tag.get_text(strip=True):
                    league = league_tag.get_text(strip=True)
                elif league_tag:
                    league = league_tag.get('href').split('/')[-1].upper().replace('-', ' ')
                else:
                    fallback = container.find_previous('a', href=lambda href: href and '/league/' in href)
                    league = fallback.get_text(strip=True) if fallback else "Unknown League"

                game_rows = container.find_all('div', class_='event--wrapp')

                for row in game_rows:
                    try:
                        time_div = row.find('div', class_='event__info--time')
                        event_time = time_div.find('time').get_text(strip=True) if time_div and time_div.find('time') else "Time Not Found"

                        match_info = row.find('div', class_='event__matchInfo')
                        match_link = match_info.find('a') if match_info else None
                        teams = match_link.get_text(separator=" ", strip=True) if match_link else ""
                        
                        if not teams and match_link:
                            slug = match_link.get('href', '')
                            teams = slug.split('/')[-1].rsplit('-', 1)[0].replace('-', ' ').title()

                        channels = []
                        channel_list = row.find('ul', class_='event__tags')
                        
                        if channel_list:
                            for channel_link in channel_list.find_all('a'):
                                channel_name = channel_link.get('aria-label')
                                if channel_name:
                                    channels.append(channel_name)

                        all_games.append({
                            "Date": game_date,
                            "Time": event_time,
                            "League": league,
                            "Matchup": teams or "Unknown Matchup",
                            "Services": ", ".join(channels)
                        })
                        
                    except Exception as e:
                        continue
            
            print(f"Finished {game_date}.\n")

        # Close the browser once the loop finishes
        await browser.close()
        print("Finished scrolling all 7 days. Parsing data...")

    # 3. Export the Data
    df = pd.DataFrame(all_games)
    print(df.head())
    df.to_csv("livesports_schedule_7days.csv", index=False)
    print(f"\nSuccessfully pulled {len(all_games)} games across 7 days!")

# Run the function
await scrape_livesports()

Launching browser...
Fetching calendar links...
Found 7 days to scrape. Starting loop...

--- Scraping Day 1 of 7: https://www.livesportsontv.com/ ---
Extracting games for date: 2/23/2026
Finished 2/23/2026.

--- Scraping Day 2 of 7: https://www.livesportsontv.com/date/tomorrow ---
Extracting games for date: 2/24/2026
Finished 2/24/2026.

--- Scraping Day 3 of 7: https://www.livesportsontv.com/date/2026-02-25 ---
Extracting games for date: 2/25/2026
Finished 2/25/2026.

--- Scraping Day 4 of 7: https://www.livesportsontv.com/date/2026-02-26 ---
Extracting games for date: 2/26/2026
Finished 2/26/2026.

--- Scraping Day 5 of 7: https://www.livesportsontv.com/date/2026-02-27 ---
Extracting games for date: 2/27/2026
Finished 2/27/2026.

--- Scraping Day 6 of 7: https://www.livesportsontv.com/date/2026-02-28 ---
Extracting games for date: 2/28/2026
Finished 2/28/2026.

--- Scraping Day 7 of 7: https://www.livesportsontv.com/date/2026-03-01 ---
Extracting games for date: 3/1/2026
Finished 3/